# Analyse de la mesure de transmission du tCBP

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pickle
from astropy.constants import e, h, c
from scipy.interpolate import interp1d
import pandas as pd
from datetime import datetime

import utils as ut # My utility functions

%matplotlib widget

In [ ]:
lsst_dir = '/home/lmousset/Projets_Recherche/LSST/'

## Thierry and Kelian data

In [ ]:
wl_thierry, tr_thierry, tr_err_thierry = ut.load_thierry_measurements()
wl_kelian, tr_kelian, tr_err_kelian = ut.load_kelian_measurements()

plt.figure(figsize=(8, 5))
plt.errorbar(wl_thierry, tr_thierry, yerr=tr_err_thierry, c='b', marker='.', ls="", label="Thierry's measurements")
plt.xlabel('Wavelength [nm]')
plt.ylabel('Transmission')
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.errorbar(wl_kelian, tr_kelian, yerr=tr_err_kelian, c='r', marker='.', ls="", label="Kelian's measurements")
plt.xlabel('Wavelength [nm]')
plt.ylabel('Transmission')
plt.legend()
plt.show()

## Photodiode efficiencies

In [ ]:
wls_interp = np.arange(300, 1150, 4)

# Get the interpolation functions for the photodiode responses
interp_func_NIST, interp_func_err_NIST = ut.interp_photodiode_response(lsst_dir + 'Mesure_transmission_tcbp_avec_LEDs/NIST_e_per_photon.npy') # e/photon
interp_func_pd8D057, interp_func_err_pd8D057 = ut.interp_photodiode_response(lsst_dir + 'Mesure_transmission_tcbp_avec_LEDs/photodiode_Qeff_8D057.npy') #e/photon

# Apply the interpolation functions to the desired wavelength grid
Q_NIST_interp = interp_func_NIST(wls_interp)
Q_NIST_interp_err = interp_func_err_NIST(wls_interp)
Q_pd8D057_interp = interp_func_pd8D057(wls_interp)
Q_pd8D057_interp_err = interp_func_err_pd8D057(wls_interp)
 
plt.figure(figsize=(10, 6))
plt.errorbar(wls_interp, Q_NIST_interp, yerr=Q_NIST_interp_err, label='NIST', c='r', marker='.', ls="")
plt.errorbar(wls_interp, Q_pd8D057_interp, yerr=Q_pd8D057_interp_err, label='Marc 8D057', c='g', marker='.', ls="")
plt.legend()
plt.xlabel('Wavelength [nm]')
plt.ylabel('Efficiency [e/photon]')
plt.title('Photodiode Calibration Data')

## Lens transmission

On récupère la mesure moyenne faite dans `optical_bench_lens_transmission.ipynb` ainsi que celle d'Elana.

In [ ]:
### Elana lens measurement
csv_file = lsst_dir + '/Mesure_transmission_tcbp_avec_LEDs/lens_ratio_calibration_system_2.csv'

wls_elana, transmission_elana, error_elana = ut.read_Elana_lens_measurements(csv_file)

### Mean measurements of the lens transmission
wl_lens, tr_lens, tr_err_lens = np.load(lsst_dir + 'tCBP_optical_bench/lens_transmission_avg_ratio_2026-11-18.npy', allow_pickle=True)
wl_lens = np.asarray(wl_lens, dtype=float)
tr_lens = np.asarray(tr_lens, dtype=float)
tr_err_lens = np.asarray(tr_err_lens, dtype=float)

# Interpolate the lens transmission
interpolator_lens = interp1d(wl_lens, tr_lens, kind='linear', fill_value='extrapolate')
interpolator_lens_err = interp1d(wl_lens, tr_err_lens, kind='linear', fill_value='extrapolate')

wls_lens_interp = np.arange(300, 1150, 4)
transmission_lens_interp = interpolator_lens(wls_lens_interp)
transmission_lens_interp_err = interpolator_lens_err(wls_lens_interp)

# Plot
plt.figure(figsize=(10,6))
plt.errorbar(wls_elana, transmission_elana, yerr=error_elana, color='k', marker='.', ls="")
plt.errorbar(wl_lens, tr_lens, yerr=tr_err_lens, color='b', marker='.', ls="")
plt.errorbar(wls_lens_interp, transmission_lens_interp, yerr=transmission_lens_interp_err, color='r', marker='.', ls='')
plt.xlabel('Wavelength [nm]')
plt.ylabel('Transmission [No unit]')
plt.title('Lens Elana Transmission')
plt.ylim(0, 1)
plt.show()

# Plot all the runs

In [ ]:
main_repo = lsst_dir + 'tCBP_optical_bench/data/'

data_dict = {'20260627_213502': ['pinhole5mm, slit20, Tout premier run'],        
             '20260628_193214': ['pinhole5mm, slit20, Papier écrasé               '],
             '20260629_134742': ['pinhole5mm, slit20, Papier écrasé (sampling max)'],
             '20260629_152137': ['pinhole5mm, slit20, Papier écrasé (sampling max)'],
             '20260630_195101': ['pinhole5mm, slit20, Après avoir refait le foyer sur le vert'], # Best
             '20260701_000936': ['pinhole5mm, slit20, Idem que Run 195101'],                     # Best
             '20260630_175411': ['pinhole5mm, slit20, Foyer sur le bleu'],                       # Best
             '20260628_214053': ['pinhole1mm, slit20, '],
             '20260629_004614': ['pinhole200µm, slit20, Trou de 200µm, bruité mais ok'],
             '20260629_170826': ['pinhole5mm, slit20, ZTF-like'],
             '20260629_194844': ['pinhole200µm, slit20, ZTF-like'],
             '20260629_213351': ['pinhole5mm, slit20, StarDice-like'],
             '20260701_140913': ['pinhole5mm, slit20, ZTF-like'],
             '20260708_002717': ['pinhole5mm, slit20, Marc photodiode 8D057'],
             '20260708_013122': ['pinhole5mm, slit20, Marc photodiode 8D057 StarDice-like']
             }                                  

# Number of runs
nruns = len(data_dict.keys())


# Boucle sur les runs
for i, key in enumerate(data_dict.keys()):
    print(f'Run {i}: {key}')
    
    # Récupère le fichier run.py
    matching_dirs = [d for d in os.listdir(main_repo) if d.startswith(key)]
    data_dir = os.path.join(lsst_dir + 'tCBP_optical_bench/data/', matching_dirs[0])
    run = np.load(data_dir + "/run.npy", allow_pickle=True)

    # Get the ratio and its error
    wls_run, ratio, ratio_err = ut.get_data_run(run, doplot=False)

    ### Append to the data_dict
    data_dict[key].append(wls_run)
    data_dict[key].append(ratio)
    data_dict[key].append(ratio_err)

In [ ]:
## Plot all ratios
plt.figure(figsize=(14,7))
for i, key in enumerate(data_dict.keys()):
    color = cm.plasma(i / nruns)
    print(f'Run {i}: {key}')
    # Normalization
    #norm = all_ratio[i][all_wls_run_sorted[i]==769.996][0]
    norm = data_dict[key][2][data_dict[key][1]==500.][0]
    plt.errorbar(data_dict[key][1], data_dict[key][2] / norm, yerr=np.abs(data_dict[key][3] / norm), color=color, marker='.', ls="", label=f'Run {i} - {data_dict[key][0]}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
plt.ylim(0, 1.5)
plt.legend(fontsize=8)

# Make a selection of runs

In [ ]:
selected_runs = ['20260630_195101', '20260701_000936', '20260630_175411']
nruns_selected = len(selected_runs)

all_wls_run_sorted = []
all_ratio = []
all_ratio_err = []
for i, key in enumerate(selected_runs):
    all_wls_run_sorted.append(data_dict[key][1])
    all_ratio.append(data_dict[key][2])
    all_ratio_err.append(data_dict[key][3])

# Plot selected runs
plt.figure(figsize=(10,5))
for i, key in enumerate(selected_runs):
    print(f'Run {i}: {key}')
    color = cm.plasma(i / nruns_selected)
    # Normalization
    #norm = all_ratio[i][all_wls_run_sorted[i]==769.996][0]
    norm = all_ratio[i][all_wls_run_sorted[i]==500.][0]
    plt.errorbar(all_wls_run_sorted[i], all_ratio[i] / norm, yerr=np.abs(all_ratio_err[i] / norm), color=color, marker='.', ls="", 
                 label=f'Run {i} - {data_dict[key][0]}') 
    plt.xlabel('wavelengths [nm]')
    plt.ylabel('ratio Receiver/Sphere')
    plt.grid(True)  
    plt.legend(fontsize=8)


In [ ]:
# Rapport entre 2 scans
irun1 = 2
irun2 = 1

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio[irun1][indices1]
commun_ratio2 = all_ratio[irun2][indices2]

commun_ratio_err1 = all_ratio_err[irun1][indices1]
commun_ratio_err2 = all_ratio_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=rapport_err, c='orange', marker='.', ls="")
plt.ylim(0.99, 1.01)
plt.ylabel('ratio entre deux runs')

## Apply the correction factor : Photodiode and Lens 

In [ ]:
### Correction factor
all_ratio_corrected = []
all_ratio_corrected_err = []
for i, key in enumerate(selected_runs):
    wls = all_wls_run_sorted[i]
    transmission_lens_interp = interpolator_lens(wls)
    transmission_lens_interp_err = interpolator_lens_err(wls) 

    if i in [10]:
        print(f"Run {i} is using 8D057 photodiode")
        pd_interp_e_per_photon = interp_func_pd8D057(wls)
        pd_interp_e_per_photon_err = interp_func_err_pd8D057(wls)
    
    else :
        print(f"Run {i} is using NIST photodiode")
        pd_interp_e_per_photon = interp_func_NIST(wls)
        pd_interp_e_per_photon_err = interp_func_err_NIST(wls)

    #print(f"Run {i}: pd_interp_e_per_photon = {pd_interp_e_per_photon}")

    correction_factor = 1 / (pd_interp_e_per_photon * transmission_lens_interp * e.value) 
    ratio_corrected = all_ratio[i] * correction_factor

    ### Erreurs
    ratio_corrected_err = np.sqrt((all_ratio_err[i] / all_ratio[i])**2 + 
                                  (pd_interp_e_per_photon_err / pd_interp_e_per_photon)**2 + 
                                  (transmission_lens_interp_err / transmission_lens_interp)**2) * ratio_corrected
    
    all_ratio_corrected.append(ratio_corrected)
    all_ratio_corrected_err.append(ratio_corrected_err)

In [ ]:
plt.figure(figsize=(10,5))
for i, key in enumerate(selected_runs):
    #norm = all_ratio_corrected[i][all_wls_run_sorted[i]==500.][0]
    #print(norm)
    color = cm.plasma(i / nruns_selected)
    plt.errorbar(all_wls_run_sorted[i], all_ratio_corrected[i], yerr=np.abs(all_ratio_corrected_err[i]), color=color,marker='.', ls="-", 
                 label=f'Run {selected_runs[i]} - {data_dict[selected_runs[i]][0]}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere corrected')
plt.ylim(0, 1e18)
plt.grid(True)
plt.legend()

# Moyenner les runs sélectionnés



In [ ]:
nruns_selected = len(selected_runs)
# Trouver TOUTES les wavelengths uniques
all_unique_wls_runs = np.array([])
for i in range(len(selected_runs)):
    all_unique_wls_runs = np.union1d(all_unique_wls_runs, all_wls_run_sorted[i])

print(f"Toutes les wavelengths uniques des runs: {len(all_unique_wls_runs)} points, de {all_unique_wls_runs.min():.1f} à {all_unique_wls_runs.max():.1f} nm")

In [ ]:
# Pour chaque wavelength, compter combien de runs l'ont et la moyenner
avg_ratio_corrected = []
avg_ratio_corrected_err = []
count_per_wl_runs = []

for wl in all_unique_wls_runs:
    ratios = []
    ratio_errs = []
    
    for i in range(nruns_selected):
        wls = all_wls_run_sorted[i]
        ratio = all_ratio_corrected[i]
        ratio_err = all_ratio_corrected_err[i]
        
        # Chercher si cette wavelength existe dans ce run
        idx = np.where(np.isclose(wls, wl, rtol=1e-9))[0]
        if len(idx) > 0:
            ratios.append(ratio[idx[0]])
            ratio_errs.append(ratio_err[idx[0]])
    
    count = len(ratios)
    if count > 0:
        avg_ratio_corrected.append(np.mean(ratios))
        
        if count > 1:
            # Calculer le STD sur les runs
            err = np.std(ratios, ddof=1) / np.sqrt(count)
        else:
            # Un seul run : garder l'erreur du run
            err = ratio_errs[0]
        
        avg_ratio_corrected_err.append(err)
        count_per_wl_runs.append(count)

avg_ratio_corrected = np.array(avg_ratio_corrected)
avg_ratio_corrected_err = np.array(avg_ratio_corrected_err)
count_per_wl_runs = np.array(count_per_wl_runs)


In [ ]:
print(f"\nMoyenne des runs: {len(avg_ratio_corrected)} points")

print(f"  - {np.sum(count_per_wl_runs == nruns_selected)} points avec {nruns_selected} runs")
for n in range(nruns_selected-1, 0, -1):
    count_n = np.sum(count_per_wl_runs == n)
    if count_n > 0:
        print(f"  - {count_n} point(s) avec {n} run(s)")
print(f"Ratio moyen: {avg_ratio_corrected.min():.3f} à {avg_ratio_corrected.max():.3f}")

In [ ]:
# Tracer
plt.figure(figsize=(10,6))
for i in range(nruns_selected):
    color = cm.plasma(i / nruns_selected)
    plt.errorbar(all_wls_run_sorted[i], all_ratio_corrected[i], yerr=np.abs(all_ratio_corrected_err[i]), color=color,
                 marker='.', ls="", label=f'Run {selected_runs[i]} - {data_dict[selected_runs[i]][0]}')

plt.errorbar(all_unique_wls_runs, avg_ratio_corrected, yerr=avg_ratio_corrected_err, 
             marker='o', ls="-", linewidth=2, markersize=1, color='k', 
             label=f'Moyenne des {nruns_selected} runs')

plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio (corrected)")
plt.legend(fontsize=8)
plt.ylim(0, 1e18)
plt.grid()
plt.show()

In [ ]:
# Rapport entre 2 scans après avoir appliqué le facteur de correction
irun1 = 2
irun2 = 1

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])
print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio_corrected[irun1][indices1]
commun_ratio2 = all_ratio_corrected[irun2][indices2]

commun_ratio_err1 = all_ratio_corrected_err[irun1][indices1]
commun_ratio_err2 = all_ratio_corrected_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=np.abs(rapport_err), c='orange', marker='.', ls="")
plt.ylim(0.9, 1.05)
plt.ylabel('ratio entre deux runs')

# Save the averaged measurement

In [ ]:
# Save the average ratio and its error to a file
output_file = lsst_dir + f'tCBP_optical_bench/tcbp_averaged_transmission_{datetime.now().strftime('%Y-%m-%d')}.npy'

my_tcbp = np.zeros((3, len(all_unique_wls_runs)))
my_tcbp[0,:] = all_unique_wls_runs
my_tcbp[1,:] = avg_ratio_corrected
my_tcbp[2,:] = avg_ratio_corrected_err

# Transformer ce tableau en record array
np.save(output_file, np.core.records.fromarrays(my_tcbp, names='cbp_wl, cbp_tr, cbp_tr_err', formats='f8,f8,f8'))
